# 02 EModel Optimisation

Build an `EModelOptimizationScanConfig`, expand it via `GridScanGenerationTask`, and
run the `EModelOptimizationTask` for each coordinate. The task downloads the extraction
`TaskResult` assets (ephys data + `extracted_features.json` + recipes) from entitycore,
stages the morphology SWC and ion channel model `.mod` files from entity entities,
compiles the mod files, merges the optimisation-related `pipeline_settings` from the blocks
into `recipes.json`, then runs `pipeline.optimise(seed=...)` followed by analysis and
draft emodel export.

**Reads from:** the entitycore staging project — the extraction `TaskResult` entity
from notebook 01, plus `CellMorphology` and `IonChannelModel` entities.

**Writes to:** `obi-output/02_emodel_optimization/grid_scan/0/` — the working directory
plus the new `checkpoints/` directory and the `run/` archive.

The example uses **`optimiser='SO-CMA'`** with very small `max_ngen=2` and
`offspring_size=4`. Increase those for production runs.

## Imports

In [ ]:
import json
from pathlib import Path

import obi_one as obi
from entitysdk import Client, ProjectContext
from obi_auth import get_token
from obi_one.core.info import Info
from obi_one.scientific.from_id.etype_class_from_id import ETypeClassFromID
from obi_one.scientific.from_id.ion_channel_model_from_id import IonChannelModelFromID
from obi_one.scientific.from_id.task_result_from_id import TaskResultFromID
from obi_one.scientific.tasks.emodel_building.task2_emodel_optimization.blocks import (
    OptimizationInitialize,
    OptimizationParams,
    OptimizationSettings,
    ParametersSelection,
)


## Connect to entitycore staging

The optimisation task downloads the extraction TaskResult assets and entity
morphology/mechanisms from entitycore. We use the staging environment + test project.

In [ ]:
token = get_token(environment="staging")
project_context = ProjectContext(
    virtual_lab_id=obi.LAB_ID_STAGING_TEST,
    project_id=obi.PROJECT_ID_STAGING_TEST,
)
db_client = Client(
    api_url="https://staging.cell-a.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)
print("Connected to entitycore staging.")

## Build the scan config

The optimisation stage uses entity-based inputs:
- `target_efeatures`: the extraction `TaskResult` entity from notebook 01
- `morphology`: a `CellMorphology` entity (SWC/ASC asset is downloaded + staged)
- `ion_channel_models`: `IonChannelModel` entities (`.mod` assets are downloaded + staged).
  The params file is built dynamically from these models.

In [ ]:
# Replace with real entity IDs from your staging project.
EXTRACTION_TASK_RESULT_ID = "38362e43-04c2-4c1d-8e21-18d7a3750eea"
ETYPE_ID = "75703892-27a4-4588-9f12-145b08051db5"  # Staging: cADpyr
MORPHOLOGY_ID = "de34e33d-594d-4227-83d1-4e5bb657cf0c" # C060114A5, species: rat
ION_CHANNEL_MODEL_IDS = [
    "3d371a7e-ff6a-45a0-9f32-34505ac940df", #"name": "CaDynamics_DC0",
    "21ebb7ab-b41b-441d-b494-6665075d26b0", #"name": "Ca_HVA2",
    "91cfc99a-9dc7-462c-9b08-330dfb4db871", #"name": "Ca_LVAst",
    "af9eef69-40a3-4e41-9c3f-542a9ff689fc", #"name": "Ih",
    "bdc5d10e-fd25-403b-83cd-81853676ca35", #"name": "K_Pst",
    "8c60dccf-8dfc-48a4-a2db-358b9643e3fb", #"name": "K_Tst",
    "7decbf75-9414-4754-bf54-5fa787097206", #"name": "Nap_Et2",
    "13c947c3-cb76-4a9a-91f4-146e95bd25f3", #"name": "NaTg",
    "e1090418-c607-4901-88ae-a5ceb5bfb75a", #"name": "SK_E2",
    "a014a0f6-217a-4e18-935f-aa298e35c9d1", #"name": "SKv3_1",
]

scan_config = obi.EModelOptimizationScanConfig(
    info=Info(
        campaign_name="L5PC Optimisation",
        campaign_description="Optimise L5PC model against extracted e-features.",
    ),
    initialize=OptimizationInitialize(
        target_efeatures=TaskResultFromID(id_str=EXTRACTION_TASK_RESULT_ID),
        emodel="L5PC",
        morphology=obi.CellMorphologyFromID(id_str=MORPHOLOGY_ID),
        etype=ETypeClassFromID(id_str=ETYPE_ID),
    ),
    parameters_selection=ParametersSelection(
        ion_channel_models=[
            IonChannelModelFromID(id_str=icm_id)
            for icm_id in ION_CHANNEL_MODEL_IDS
        ],
    ),
    optimization_settings=OptimizationSettings(
        optimiser="SO-CMA",
        max_ngen=1,            # very small for demo runs
        optimisation_timeout=300.0,
        validation_threshold=5.0,
        seed=1,
    ),
    optimization_params=OptimizationParams(
        offspring_size=1,      # very small for demo runs
    ),
)
print(scan_config.optimization_settings.to_dict(scan_config.optimization_params))

## Run the grid scan

Per-coordinate output goes to `obi-output/02_emodel_optimization/grid_scan/<idx>/`.

In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/02_emodel_optimization/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute(db_client=db_client)

#  Uncomment to run the grid scan
print(f"Generated {len(grid_scan.single_configs)} single configs.")
print("To execute the grid scan, uncomment the line below.")
# obi.run_tasks_for_generated_scan(grid_scan, db_client=db_client)

In [ ]:
# Print entity IDs
print("=== Entity IDs ===")
print(f"Extraction TaskResult (input): {EXTRACTION_TASK_RESULT_ID}")
print(f"Campaign TaskConfig:           {grid_scan.form.campaign.id}")
for i, sc in enumerate(grid_scan.single_configs):
    print(f"Single TaskConfig [{i}]:          {sc.single_entity.id}")
print(f"Morphology:                    {MORPHOLOGY_ID}")
print(f"Ion Channel Models:            {ION_CHANNEL_MODEL_IDS}")

## Inspect the checkpoints

In [ ]:
coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print("Coordinate output:", coord_root)

checkpoints = sorted((coord_root / "checkpoints").rglob("*.pkl"))
print(f"Checkpoint files ({len(checkpoints)}):")
for p in checkpoints:
    print(" ", p.relative_to(coord_root))